<a href="https://colab.research.google.com/github/maggiecrowner/DS5001-Final-Project/blob/main/VOCAB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Parsed and Annotated Data

In [1]:
! git clone https://github.com/maggiecrowner/DS5001-Final-Project.git

Cloning into 'DS5001-Final-Project'...
remote: Enumerating objects: 263, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 263 (delta 48), reused 4 (delta 4), pack-reused 175 (from 1)
Receiving objects: 100% (263/263), 15.04 MiB | 8.80 MiB/s, done.
Resolving deltas: 100% (99/99), done.


In [2]:
!wget -O CORPUS.csv "https://virginia.box.com/shared/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv"

--2026-04-21 02:07:40--  https://virginia.box.com/shared/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Resolving virginia.box.com (virginia.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to virginia.box.com (virginia.box.com)|74.112.186.157|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv [following]
--2026-04-21 02:07:40--  https://virginia.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Reusing existing connection to virginia.box.com:443.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://virginia.app.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv [following]
--2026-04-21 02:07:41--  https://virginia.app.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Resolving virginia.app.box.com (virginia.app.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to virginia.app.box.com (virginia.app.box.com)|74.112.186.157|:443.

In [3]:
import pandas as pd
CORPUS = pd.read_csv('CORPUS.csv', delimiter='|')

## VOCAB

In [4]:
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import numpy as np

ps = PorterStemmer()

n = CORPUS['term_str'].value_counts().rename('n')
doc_term = CORPUS.reset_index().groupby('Title')['term_str'].apply(set)
N_docs = len(doc_term)

df_counts = (CORPUS.reset_index()
             .groupby('term_str')['Title']
             .nunique()
             .rename('df'))

VOCAB = pd.DataFrame({'n': n, 'df': df_counts})
VOCAB.index.name = 'term_str'

VOCAB['p']            = VOCAB['df'] / N_docs
VOCAB['i']            = np.log2(N_docs / VOCAB['df'])
VOCAB['dfidf']        = VOCAB['df'] * VOCAB['i']
VOCAB['porter_stem']  = VOCAB.index.map(lambda t: ps.stem(t))
VOCAB['max_pos']      = CORPUS.groupby('term_str')['pos'].agg(lambda x: x.value_counts().idxmax())
VOCAB['max_pos_group']= CORPUS.groupby('term_str')['pos_group'].agg(lambda x: x.value_counts().idxmax())
VOCAB['stop']         = VOCAB.index.map(lambda t: int(t in ENGLISH_STOP_WORDS))
VOCAB['ngram_length'] = VOCAB.index.map(lambda t: len(t.split()))
VOCAB = VOCAB.drop(columns=['df'])

print(f" VOCAB shape: {VOCAB.shape}")
VOCAB.head()

 VOCAB shape: (19102, 9)


,n,p,i,dfidf,porter_stem,max_pos,max_pos_group,stop,ngram_length
term_str,,,,,,,,,
0,151,0.030345,5.042406,443.731690,0,CD,OTHER,0,1
00,33,0.008621,6.857981,171.449525,00,CD,OTHER,0,1
000,14,0.002069,8.916875,53.501248,000,CD,OTHER,0,1
000s,1,0.000345,11.501837,11.501837,000,CD,OTHER,0,1
004,1,0.000345,11.501837,11.501837,004,CD,OTHER,0,1


In [5]:
VOCAB.to_csv('/content/DS5001-Final-Project/VOCAB.csv', sep='|', index=True)

In [6]:
print("Number of observations/vocab:", len(VOCAB['dfidf']))

Number of observations/vocab: 19102


In [7]:
top20 = (VOCAB[VOCAB['stop'] == 0]
         .sort_values('dfidf', ascending=False)
         .head(20))
print("Top 20 significant words by DFIDF:")
top20[['n', 'p', 'i', 'dfidf', 'porter_stem', 'max_pos_group']]

Top 20 significant words by DFIDF:


,n,p,i,dfidf,porter_stem,max_pos_group
term_str,,,,,,
say,3023,0.368966,1.438442,1539.133051,say,VERB
baby,5378,0.380000,1.395929,1538.313401,babi,NOUN
time,2992,0.385172,1.376424,1537.465289,time,NOUN
want,4758,0.350000,1.514573,1537.291770,want,VERB
yeah,6556,0.387241,1.368695,1537.044454,yeah,NOUN
pre,2130,0.347586,1.524557,1536.753719,pre,NOUN
youre,4001,0.401724,1.315723,1532.817231,your,NOUN
got,4441,0.404828,1.304620,1531.624457,got,VERB
make,2513,0.324483,1.623786,1527.982882,make,VERB
